In [3]:
import pandas as pd
import os
import time
import re
from pathlib import Path
import pandasai as pai
from pandasai_litellm.litellm import LiteLLM
from litellm import completion

In [4]:
df_clinic_level = pd.read_csv("cc_clinic_level.csv")
df_doctor = pd.read_csv("cc_doctor.csv")
df_hourly = pd.read_csv("cc_hourly.csv")
df_patient = pd.read_csv("cc_patient.csv")

df_clinic_level = pai.DataFrame(df_clinic_level)
df_doctor = pai.DataFrame(df_doctor)
df_hourly = pai.DataFrame(df_hourly)
df_patient = pai.DataFrame(df_patient)

In [5]:
def generate_Response(question: str, dataframes: list, model_id: str = None, api_key: str = None):
    """
    Generate a response using PandasAI for the given question.
    
    Args:
        question: The question to ask
        dataframes: List of PandasAI DataFrames to query
        model_id: The LLM model identifier
        api_key: API key for the LLM
    
    Returns:
        tuple: (response, elapsed_time)
    """
    default_api_key = "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"
    default_model = "nvidia_nim/mistralai/mistral-large-3-675b-instruct-2512"
    
    # Configure the LLM
    llm = LiteLLM(
        model=model_id or default_model,
        api_key=api_key or default_api_key,
        stream=False,
    )
    pai.config.set({"llm": llm, "save_charts": False})
    
    start_time = time.time()
    response = pai.chat(question, *dataframes)
    elapsed_time = time.time() - start_time
    
    return response, elapsed_time

In [6]:
def extract_Generated_code(log_path: str = "pandasai.log"):
    """
    Extract the full generated code from PandasAI log.
    
    Returns:
        dict: {"sql_query": str, "full_code": str}
    """
    from pathlib import Path
    
    log_file = Path(log_path)
    if not log_file.exists():
        return {"sql_query": "", "full_code": ""}
    
    text = log_file.read_text(encoding="utf-8")
    
    # --- Extract full generated code from "Executing code:" section ---
    full_code = ""
    # Pattern to find code after "Executing code:" until next log entry or end
    exec_pattern = r'\[INFO\] Executing code:\s*(.+?)(?=\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2} \[INFO\]|$)'
    exec_matches = re.findall(exec_pattern, text, re.DOTALL)
    if exec_matches:
        full_code = exec_matches[-1].strip()
    
    # --- Extract SQL query from the full code ---
    sql_query = ""
    # Pattern for triple-quoted multi-line SQL
    multi_line_pattern = r'sql_query\s*=\s*"""(.+?)"""'
    single_quote_pattern = r"sql_query\s*=\s*'([^']+)'"
    double_quote_pattern = r'sql_query\s*=\s*"([^"]+)"'

    # Search in full_code first, then in entire text
    search_text = full_code if full_code else text
    
    matches = re.findall(multi_line_pattern, search_text, re.DOTALL)
    if not matches:
        matches = re.findall(single_quote_pattern, search_text)
    if not matches:
        matches = re.findall(double_quote_pattern, search_text)
    if matches:
        sql_query = matches[-1].strip()
    
    return {"sql_query": sql_query, "full_code": full_code}


def clear_log(log_path: str = "pandasai.log"):
    """Clear the PandasAI log file after extraction."""
    from pathlib import Path
    log_file = Path(log_path)
    if log_file.exists():
        log_file.write_text("", encoding="utf-8")

In [7]:
def validation_Response(query: str, question: str,response=None,answer=None) -> bool:
    if not query:
        return False
    
    try:
        validation = completion(
            model="nvidia_nim/meta/llama-3.1-405b-instruct",
            messages=[{
                "role": "user", 
                "content": f"""You are a query validator.
                Question: {question}
                Code: {query}
                response: {response} if response is chart or visualization, compare answer with the query else compare response with the answer
                answer: {answer}
                Does this code correctly retrieve the data required to answer the question?
                Respond with ONLY one word: TRUE or FALSE."""
            }],
            api_key= "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"
            
        )
        return validation.choices[0].message.content.strip()
        
    except Exception as e:
        print(f"⚠️ Validation error: {e}")
        return None

In [9]:
def error_explaination_Response(query: str, question: str, response=None, answer=None) -> str:
    if not query:
        return "No code generated to validate."
    
    try:
        explanation = completion(
            model="nvidia_nim/meta/llama-3.1-405b-instruct",
            messages=[{
                "role": "user", 
                "content": f"""You are a helpful assistant that explains errors in code.
                Question: {question}
                Code: {query}
                Response: {response}
                Answer: {answer}
                the code is determine by the llms judge passed to you thats means it is wrong, explain what is wrong with it and how to fix it. If the code is correct, say "The code is correct."."""
            }],
            api_key= "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"
        )
        return explanation.choices[0].message.content.strip()
        
    except Exception as e:
        print(f"⚠️ Explanation error: {e}")
        return "An error occurred while generating the explanation."

In [25]:
clear_log()
# Simple test using the helper functions
question = "Which clinic serves the highest number of patients from different countries, and which doctor handles the majority of those patients?"

# Use generate_Response function
response, elapsed_time = generate_Response(question, [df_doctor, df_hourly, df_patient,df_clinic_level])
print(f"Response: {response.value if hasattr(response, 'value') else response}")

# Extract full generated code
generated = extract_Generated_code()
print(f"\nSQL Query:\n{generated['sql_query']}")
print(f"\nFull Code:\n{generated['full_code']}")

# Use validation_Response function with full_code for complete validation
code_to_validate = generated['full_code'] if generated['full_code'] else generated['sql_query']
is_correct = validation_Response(code_to_validate, question, response=response, answer=""""DOCTOR HE treats 1,996 patients at CLINIC T.""")
print(f"\nIs Correct: {is_correct}")

# Check if validation returned "FALSE" (string comparison, case-insensitive)
if is_correct and "false" in str(is_correct).lower():
    explanation = error_explaination_Response(code_to_validate, question, response,answer)
    print(f"\nError Explanation:\n{explanation}")

print(f"Time taken: {elapsed_time:.2f} seconds")

NoResultFoundError: No result was returned from the code execution. Please return the result in dictionary format, for example: result = {'type': ..., 'value': ...}

In [10]:
clear_log()
# Simple test using the helper functions
question = "Which clinic has the highest TotalRevenue?"

# Use generate_Response function
response, elapsed_time = generate_Response(question, [df_doctor, df_hourly, df_patient,df_clinic_level])
print(f"Response: {response.value if hasattr(response, 'value') else response}")

# Extract full generated code
generated = extract_Generated_code()
print(f"\nSQL Query:\n{generated['sql_query']}")
print(f"\nFull Code:\n{generated['full_code']}")

# Use validation_Response function with full_code for complete validation
code_to_validate = generated['full_code'] if generated['full_code'] else generated['sql_query']
is_correct = validation_Response(code_to_validate, question, response=response)
print(f"\nIs Correct: {is_correct}")

# Check if validation returned "FALSE" (string comparison, case-insensitive)
if is_correct and "false" in str(is_correct).lower():
    explanation = error_explaination_Response(code_to_validate, question, response)
    print(f"\nError Explanation:\n{explanation}")

print(f"Time taken: {elapsed_time:.2f} seconds")

Response: The clinic with the highest TotalRevenue is Clinic I with a TotalRevenue of 3,487,368.95.

SQL Query:
SELECT
    IDOrganisation as Clinic,
    TotalRevenue
FROM
    table_bd4f8d5ffa8239c09831e5b997bc2753
ORDER BY
    TotalRevenue DESC
LIMIT 1

Full Code:
import pandas as pd
sql_query = """
SELECT
    IDOrganisation as Clinic,
    TotalRevenue
FROM
    table_bd4f8d5ffa8239c09831e5b997bc2753
ORDER BY
    TotalRevenue DESC
LIMIT 1
"""
df = execute_sql_query(sql_query)
highest_revenue_clinic = df.iloc[0]['Clinic']
highest_revenue_amount = df.iloc[0]['TotalRevenue']
result = {'type': 'string', 'value': f'The clinic with the highest TotalRevenue is {highest_revenue_clinic} with a TotalRevenue of {highest_revenue_amount:,.2f}.'}

Is Correct: TRUE
Time taken: 8.05 seconds


In [ ]:
# Benchmark Test Questions with Expected Answers
# Each entry: {"question": str, "answer": str, "dataframes": list}

benchmark_questions = {
    "easy": [
        {
            "question": "Which clinic has the highest TotalRevenue?",
            "answer": "Clinic I",
            "dataframes": [df_clinic_level]
        },
        {
            "question": "What is the average TotalRevenue across all clinics?",
            "answer": "1,220,906.65",
            "dataframes": [df_clinic_level]
        },
        {
            "question": "List all unique Districts present in the dataset.",
            "answer": "Bandar Enstek, Bangi, Batang Kali, Batu Caves, Bayan Lepas, Central KL, Cheras, Desa Pandan, Dungun, Georgetown, Hulu Langat, Ipoh, Kajang, Kota Bharu, Kuala Langat, Kuala Lumpur, Kuantan, Kulim, Mantin, Muar, Pasir Mas, Petaling, Petaling Jaya, Presint 8, Rawang, Semenyih, Sepang, Seremban, Setia Alam, Shah Alam, Skudai, Wakaf Bharu",
            "dataframes": [df_clinic_level]
        },
        {
            "question": "Which hour has the highest patient count?",
            "answer": "2025-02-13 18:00:00",
            "dataframes": [df_hourly]
        },
        {
            "question": "Which state generates the highest total revenue?",
            "answer": "Kuala Lumpur",
            "dataframes": [df_clinic_level]
        }
    ],
    "medium": [
        {
            "question": "Which doctor treats the highest number of patients from Malay ethnicity?",
            "answer": "Doctor B",
            "dataframes": [df_doctor, df_patient]
        },
        {
            "question": "Which doctor sees the most patients from Married social status group?",
            "answer": "Doctor BZ",
            "dataframes": [df_doctor, df_patient]
        },
        {
            "question": "Which doctor has the highest number of patients from different country/geography?",
            "answer": "Doctor AW treats the most patients from Australia, Bangladesh, Canada, Germany, India, Japan, Malaysia, Philippines, and Singapore",
            "dataframes": [df_doctor, df_patient]
        },
        {
            "question": "Which clinic has the highest revenue and how many doctors work there?",
            "answer": "Clinic I with 15 doctors",
            "dataframes": [df_clinic_level, df_doctor]
        },
        {
            "question": "Which doctor treats the highest number of patients from Malay ethnicity in the highest revenue clinic?",
            "answer": "Doctor B in Clinic I",
            "dataframes": [df_clinic_level, df_doctor, df_patient]
        }
    ],
    "hard": [
        {
            "question": "Create a pie chart of patient religions for the top 3 clinics by total revenue. Analyze religious demographics within high performing clinics.",
            "answer": "Pie chart showing religious distribution",
            "dataframes": [df_clinic_level, df_patient]
        },
        {
            "question": "Plot the relationship between a clinic's total revenue and the number of doctors it employs.",
            "answer": "Scatter plot of revenue vs doctor count",
            "dataframes": [df_clinic_level, df_doctor]
        },
        {
            "question": "Create a violin plot of revenue. Separate data by Locum or Residence status. Compare earnings.",
            "answer": "Violin plot comparing Locum vs Residence revenue",
            "dataframes": [df_doctor]
        },
        {
            "question": "Create a stacked bar chart of patient religion counts by state. Visualize religious demographics in regions.",
            "answer": "Stacked bar chart of religions by state",
            "dataframes": [df_clinic_level, df_patient]
        },
        {
            "question": "Plot the clinics with the highest revenue and the top 5 high patient count doctors working there.",
            "answer": "Combined chart of top clinics and doctors",
            "dataframes": [df_clinic_level, df_doctor]
        }
    ]
}

# Models to test
models_to_test = [
    ("nvidia_nim/meta/llama-3.1-8b-instruct", "Llama-3.1-8B"),
    ("nvidia_nim/google/gemma-2-27b-it", "Gemma-2-27B"),
    ("nvidia_nim/meta/llama3-70b-instruct", "Llama3-70B"),
    ("nvidia_nim/meta/llama-3.1-405b-instruct", "Llama-3.1-405B"),
    ("nvidia_nim/mistralai/mistral-large-3-675b-instruct-2512", "Mistral-Large-3-675B")
]

print(f"📋 Benchmark configured:")
print(f"   Easy questions: {len(benchmark_questions['easy'])}")
print(f"   Medium questions: {len(benchmark_questions['medium'])}")
print(f"   Hard questions: {len(benchmark_questions['hard'])}")
print(f"   Models to test: {len(models_to_test)}")

In [ ]:
# Benchmark Runner
NUM_RUNS = 3
MAX_RETRIES = 2
RESULTS_FILE = "exports/benchmark_results_2.csv"

os.makedirs("exports", exist_ok=True)

# Load existing results if available
if os.path.exists(RESULTS_FILE) and os.path.getsize(RESULTS_FILE) > 0:
    try:
        results_df = pd.read_csv(RESULTS_FILE)
        results = results_df.to_dict('records')
        print(f"📂 Loaded {len(results)} existing results from {RESULTS_FILE}")
    except pd.errors.EmptyDataError:
        results = []
        print("📂 Results file was empty, starting fresh")
else:
    results = []
    print("📂 Starting fresh benchmark")


def get_completed_tests(results):
    """Get set of completed (model, difficulty, question, run_num) tuples"""
    return {(r['model'], r['difficulty'], r['question'], r.get('run_num', 1)) for r in results}


def save_results(results):
    """Save results to CSV"""
    df = pd.DataFrame(results)
    df.to_csv(RESULTS_FILE, index=False)
    print(f"💾 Saved {len(results)} results")


def run_single_test(model_id, model_name, question_data, difficulty, run_num):
    """Run a single benchmark test"""
    question = question_data["question"]
    expected_answer = question_data["answer"]
    dataframes = question_data["dataframes"]
    
    for attempt in range(MAX_RETRIES + 1):
        try:
            clear_log()
            
            # Generate response
            response, elapsed_time = generate_Response(question, dataframes, model_id=model_id)
            response_value = response.value if hasattr(response, 'value') else str(response)
            
            # Extract generated code
            generated = extract_Generated_code()
            sql_query = generated["sql_query"]
            full_code = generated["full_code"]
            
            # Validate with expected answer
            code_to_validate = full_code if full_code else sql_query
            is_correct = None
            if code_to_validate:
                is_correct = validation_Response(
                    code_to_validate, 
                    question, 
                    response=response_value,
                    answer=expected_answer
                )
            
            return {
                "model": model_name,
                "difficulty": difficulty,
                "question": question,
                "expected_answer": expected_answer,
                "run_num": run_num,
                "response": response_value,
                "sql_query": sql_query,
                "full_code": full_code,
                "is_correct": is_correct,
                "time_seconds": round(elapsed_time, 2),
                "success": True,
                "error": None
            }
            
        except Exception as e:
            if attempt < MAX_RETRIES:
                print(f"      ⚠️ Retry {attempt + 1}...")
                time.sleep(2)
            else:
                return {
                    "model": model_name,
                    "difficulty": difficulty,
                    "question": question,
                    "expected_answer": expected_answer,
                    "run_num": run_num,
                    "response": None,
                    "sql_query": None,
                    "full_code": None,
                    "is_correct": False,
                    "time_seconds": None,
                    "success": False,
                    "error": str(e)[:200]
                }


# Calculate totals
total_questions = sum(len(q) for q in benchmark_questions.values())
total_tests = len(models_to_test) * total_questions * NUM_RUNS
completed_tests = get_completed_tests(results)

print("\n" + "=" * 70)
print(f"🚀 BENCHMARK: {len(models_to_test)} models × {total_questions} questions × {NUM_RUNS} runs = {total_tests} tests")
print(f"📊 Already completed: {len(completed_tests)}")
print("=" * 70)

try:
    for run_num in range(1, NUM_RUNS + 1):
        print(f"\n{'='*70}")
        print(f"🔄 RUN {run_num}/{NUM_RUNS}")
        print("=" * 70)
        
        for model_id, model_name in models_to_test:
            print(f"\n🤖 Model: {model_name}")
            print("-" * 50)
            
            for difficulty in ["easy", "medium", "hard"]:
                questions = benchmark_questions[difficulty]
                print(f"\n  📊 {difficulty.upper()} ({len(questions)} questions)")
                
                for idx, q_data in enumerate(questions):
                    question = q_data["question"]
                    
                    # Skip if already done
                    if (model_name, difficulty, question, run_num) in completed_tests:
                        print(f"    ⏭️ Q{idx+1}: Skipped (done)")
                        continue
                    
                    print(f"    🔍 Q{idx+1}: {question[:45]}...")
                    
                    result = run_single_test(model_id, model_name, q_data, difficulty, run_num)
                    results.append(result)
                    
                    # Status indicator
                    if result["success"]:
                        is_true = str(result["is_correct"]).upper() == "TRUE"
                        status = "✅" if is_true else "❌"
                    else:
                        status = "💥"
                    
                    time_str = f"{result['time_seconds']}s" if result['time_seconds'] else "N/A"
                    print(f"       {status} {time_str} | Valid: {result['is_correct']}")
                    
                    save_results(results)

except KeyboardInterrupt:
    print("\n⚠️ Interrupted! Saving progress...")
    save_results(results)

except Exception as e:
    print(f"\n❌ Error: {e}")
    save_results(results)

print("\n" + "=" * 70)
print(f"✅ Benchmark complete! Results: {len(results)}")
print(f"📁 Saved to: {RESULTS_FILE}")
print("=" * 70)

# Show summary
results_df = pd.DataFrame(results)
if len(results_df) > 0:
    print("\n📊 Summary by Model:")
    summary = results_df.groupby('model').agg({
        'success': 'mean',
        'time_seconds': 'mean',
        'is_correct': lambda x: (x.astype(str).str.upper() == 'TRUE').mean()
    }).round(3)
    summary.columns = ['Success Rate', 'Avg Time (s)', 'Accuracy']
    print(summary)

In [ ]:
# Benchmark Analysis - Run after benchmark completes
import matplotlib.pyplot as plt

# Load results
results_df = pd.read_csv(RESULTS_FILE)

# 1. Accuracy by Model and Difficulty
print("=" * 70)
print("📊 BENCHMARK RESULTS ANALYSIS")
print("=" * 70)

# Convert is_correct to boolean
results_df['correct'] = results_df['is_correct'].astype(str).str.upper() == 'TRUE'

# Accuracy summary
print("\n📈 Accuracy by Model:")
accuracy_by_model = results_df.groupby('model')['correct'].mean().sort_values(ascending=False) * 100
print(accuracy_by_model.round(1).to_string())

print("\n📈 Accuracy by Difficulty:")
accuracy_by_diff = results_df.groupby('difficulty')['correct'].mean() * 100
print(accuracy_by_diff.round(1).to_string())

print("\n📈 Accuracy by Model × Difficulty:")
pivot = results_df.pivot_table(values='correct', index='model', columns='difficulty', aggfunc='mean') * 100
print(pivot.round(1).to_string())

# 2. Response Time Analysis
print("\n⏱️ Average Response Time (seconds):")
time_by_model = results_df.groupby('model')['time_seconds'].mean().sort_values()
print(time_by_model.round(2).to_string())

# 3. Success Rate (no errors)
print("\n✅ Success Rate (no errors):")
success_by_model = results_df.groupby('model')['success'].mean() * 100
print(success_by_model.round(1).to_string())

# 4. Plot results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Accuracy bar chart
accuracy_by_model.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_xlabel('Accuracy (%)')
axes[0].set_title('Accuracy by Model')

# Time bar chart
time_by_model.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_xlabel('Avg Time (s)')
axes[1].set_title('Response Time by Model')

# Difficulty comparison
pivot.T.plot(kind='bar', ax=axes[2])
axes[2].set_xlabel('Difficulty')
axes[2].set_ylabel('Accuracy (%)')
axes[2].set_title('Model Performance by Difficulty')
axes[2].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig('exports/benchmark_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📁 Analysis saved to exports/benchmark_analysis.png")